In [4]:
import os
import pandas as pd
import joblib
import warnings

warnings.filterwarnings('ignore')

In [5]:
rf_model = joblib.load('/home/pratham/Disk2/Projects/ml_projects/loan_prediction/models/rf_model.pkl')
scaler = joblib.load('/home/pratham/Disk2/Projects/ml_projects/loan_prediction/models/scaler.pkl')
le_dict = joblib.load('/home/pratham/Disk2/Projects/ml_projects/loan_prediction/models/label_encoders.pkl')

In [6]:
def predict_loan_status(application_data):
    # Convert input data to DataFrame
    df = pd.DataFrame([application_data])
    
    # 2. Feature Engineering
    df['Dependents'] = df['Dependents'].astype(str).replace('3+', '3').astype(int)
    df['Total_Income'] = df['ApplicantIncome'] + df['CoapplicantIncome']
    df['EMI'] = (df['LoanAmount'] * 1000) / df['Loan_Amount_Term']
    
    # 3. Categorical Encoding
    encode_cols = ['Gender', 'Married', 'Education', 'Self_Employed', 'Property_Area']
    for col in encode_cols:
        df[col] = le_dict[col].transform(df[col])
        
    # 4. Feature Scaling
    num_cols = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Total_Income', 'EMI']
    df[num_cols] = scaler.transform(df[num_cols])
    
    # 5. Make the Prediction
    prediction = rf_model.predict(df)
    
    status_label = le_dict['Loan_Status'].inverse_transform(prediction)[0]
    
    if status_label == 'Y':
        return "Congratulations! Your loan application is likely to be approved."
    else:
        return "Unfortunately, your loan application is likely to be rejected. Please consider improving your financial profile and reapplying."
        

In [7]:
good_candidate = {
    'Gender': 'Female',
    'Married': 'Yes',
    'Dependents': '0',
    'Education': 'Graduate',
    'Self_Employed': 'No',
    'ApplicantIncome': 8500,     # High income
    'CoapplicantIncome': 3000,
    'LoanAmount': 120,           # Relatively low loan (120k)
    'Loan_Amount_Term': 360,     # 30 years
    'Credit_History': 1.0,       # Good credit
    'Property_Area': 'Urban'
}

result = predict_loan_status(good_candidate)
print(f"Test Case 1 Result: {result}")

Test Case 1 Result: Congratulations! Your loan application is likely to be approved.


In [8]:
risky_candidate = {
    'Gender': 'Male',
    'Married': 'No',
    'Dependents': '2',
    'Education': 'Not Graduate',
    'Self_Employed': 'Yes',
    'ApplicantIncome': 2500,     # Low income
    'CoapplicantIncome': 0,
    'LoanAmount': 400,           # Very high loan (400k)
    'Loan_Amount_Term': 180,     # Short term (15 years) => High EMI
    'Credit_History': 0.0,       # Bad credit history
    'Property_Area': 'Rural'
}

result = predict_loan_status(risky_candidate)
print(f"Test Case 2 Result: {result}")

Test Case 2 Result: Unfortunately, your loan application is likely to be rejected. Please consider improving your financial profile and reapplying.
